# 2. Open RadDB object and filter

Tutorial 1 wrote an archive. This one reads it back and filters it down.

**`RadDB` is one class with two roles.**

| role | what it is |
|---|---|
| *archive-bound* | knows where an archive lives, and reads from it |
| *data-carrying*  | holds the gates you loaded, and narrows them down |

`open()` turns the first into the second. Every operation on a data-carrying
RadDB returns a **new** one, so calls chain and nothing is changed in place.

---

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import polars as pl

import raddb

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these three paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote.

FMI_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr").expanduser()
NEXRAD_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("FMI DataTrees   :", FMI_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

FMI DataTrees   : /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr
NEXRAD DataTrees: /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr
Archive         : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive


In [ ]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "FANJ" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=3067).archive(
        datatree_dir=FMI_DIR,
        time_period=("2024-06-01", "2024-06-15"),
    )
else:
    print("archive already present:", ARCHIVE_DIR)

archive already present: /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive


## 1. `open()`: reading the archive

Reading never needs a CRS: it is recovered from the archive itself.

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="FANJ")
rdf.head()

gate_id,time,DBZH,ZDR,KDP,RHOHV,PHIDP,TEMP,volume_time,radar
i64,datetime[ns],f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str
713647000000000250,2024-06-01 12:00:08.912507648,1.17,-3.06,0.0,0.405933,98.02301,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000015250,2024-06-01 12:00:08.912507648,6.1,3.24,0.0,0.268918,137.750793,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000015750,2024-06-01 12:00:08.912507648,3.62,1.11,0.0,0.689683,91.090424,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000016250,2024-06-01 12:00:08.912507648,0.1,-2.98,0.0,0.816184,91.442001,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000016750,2024-06-01 12:00:08.912507648,3.73,10.08,0.0,0.509667,102.296822,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""


`open()` narrows *before* anything is loaded — the time range, the radars and the
columns are all pushed down into the Parquet scan, so you never pay for data you
did not ask for.

In [ ]:
# Only two variables, only radar FANJ
small_df = db.open(radars="FANJ", columns=["DBZH", "ZDR"])
print(f"small_df:\tcolumns: {small_df.columns()}\nsmall_df:\tgates: {len(small_df)}")
print("-------------------------------")
# time period
day_df = db.open(radars="FANJ", time_period=("2024-06-12", "2024-06-13"))
print(f"day_df:\t\tcolumns: {day_df.columns()}\nday_df:\t\tgates: {len(day_df):,}")

small_df:	columns: ['gate_id', 'DBZH', 'ZDR', 'volume_time', 'radar']
small_df:	gates: 2568170
-------------------------------
day_df:		columns: ['gate_id', 'time', 'DBZH', 'ZDR', 'KDP', 'RHOHV', 'PHIDP', 'TEMP', 'volume_time', 'radar']
day_df:		gates: 157,465


In [ ]:
# Filters can be pushed down at open() too, so filtered-out rows are never materialised
filtered_df = db.open(radars="FANJ", filters={"var": "DBZH", "logic": ">", "threshold": 30})
print(f"before:\t{len(rdf):,} gates\t(with DBZH > 0 dBz)\nafter:\t{len(filtered_df):,}  gates\t(with DBZH > 30 dBz)")

before:	2,568,170 gates	(with DBZH > 0 dBz)
after:	220,821  gates	(with DBZH > 30 dBz)


## 2. What you are holding

The data lives in `.data` as a **polars** DataFrame. Polars is the backend
throughout RadDB (the read path, the LUT, the archive writer).

In [ ]:
print("type:\t\t", type(rdf.data))
print("name type:\t", type(rdf.data).__name__)
print("shape:\t\t", rdf.data.shape)
rdf.data.head()

type:		 <class 'polars.dataframe.frame.DataFrame'>
name type:	 DataFrame
shape:		 (2568170, 10)


gate_id,time,DBZH,ZDR,KDP,RHOHV,PHIDP,TEMP,volume_time,radar
i64,datetime[ns],f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str
713647000000000250,2024-06-01 12:00:08.912507648,1.17,-3.06,0.0,0.405933,98.02301,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000015250,2024-06-01 12:00:08.912507648,6.1,3.24,0.0,0.268918,137.750793,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000015750,2024-06-01 12:00:08.912507648,3.62,1.11,0.0,0.689683,91.090424,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000016250,2024-06-01 12:00:08.912507648,0.1,-2.98,0.0,0.816184,91.442001,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""
713647000000016750,2024-06-01 12:00:08.912507648,3.73,10.08,0.0,0.509667,102.296822,NaN,2024-06-01 12:00:02 UTC,"""FANJ"""


In [ ]:
print("radars    :", rdf.radars())
print("variables :", rdf.columns())
print("time range:", rdf.start_time(), "->", rdf.end_time())
print("lon/lat    :", [round(v, 3) for v in rdf.geographic_extent()])
print("archive CRS:", rdf.crs())  # recovered from the archive itself

radars    : ['FANJ']
variables : ['gate_id', 'time', 'DBZH', 'ZDR', 'KDP', 'RHOHV', 'PHIDP', 'TEMP', 'volume_time', 'radar']
time range: 2024-06-01 12:00:02+00:00 -> 2024-06-14 12:00:03+00:00


lon/lat    : [22.491, 31.722, 58.659, 63.149]
archive CRS: EPSG:3067


## 3. `filter()`: threshold on values

A filter is a plain dict: `{"var", "logic", "threshold"}`

In [ ]:
rain = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})
print(f"DBZH > 20: {len(rain):,} gates")

filt_df = rdf.filter(
    [
        {"var": "DBZH", "logic": ">", "threshold": 20},
        {"var": "RHOHV", "logic": ">=", "threshold": 0.98},
        {"var": "ZDR", "logic": ">", "threshold": 4},
    ],
)
print(f"DBZH > 20, RHOHV >= 0.98, ZDR > 4 : {len(filt_df):,} gates")

DBZH > 20: 704,733 gates
DBZH > 20, RHOHV >= 0.98, ZDR > 4 : 4,843 gates


## 4. `sel()`: select by label, xarray-style

Where `filter()` thresholds *values*, `sel()` selects by **coordinate**: a time, a
sweep, a range window, a longitude/latitude box. Scalars match exactly, `slice`
gives a closed interval, and a list matches any of its members.

In [ ]:
print("one sweep      :", f"{len(rdf.sel(sweep=1)):,}")
print("sweeps 1,2,3   :", f"{len(rdf.sel(sweep=[1, 2, 3])):,}")
print("range 10-50 km :", f"{len(rdf.sel(range=slice(10_000, 50_000))):,}")
print("a lon/lat box  :", f"{len(rdf.sel(lon=slice(26.6, 27.6), lat=slice(60.6, 61.2))):,}")

one sweep      : 364,289


sweeps 1,2,3   : 814,156


range 10-50 km : 846,321


a lon/lat box  : 873,507


`range`, `azimuth`, `elevation_angle`, `latitude`, `longitude`
and `altitude` are **not stored in the Parquet files** — they live once in the LUT.
`sel()` borrows the column it needs, evaluates the selection, and drops it again,
so selecting on geometry costs no storage.

In [ ]:
print("stored per gate:", rdf.columns())
print("also selectable :", ["range", "azimuth", "elevation_angle", "latitude", "longitude", "altitude", "sweep"])

narrow = rdf.sel(sweep=1, range=slice(20_000, 60_000))
print(f"\nsweep 1, 20-60 km: {len(narrow):,} gates " f"(columns unchanged: {narrow.columns() == rdf.columns()})")

stored per gate: ['gate_id', 'time', 'DBZH', 'ZDR', 'KDP', 'RHOHV', 'PHIDP', 'TEMP', 'volume_time', 'radar']
also selectable : ['range', 'azimuth', 'elevation_angle', 'latitude', 'longitude', 'altitude', 'sweep']



sweep 1, 20-60 km: 56,627 gates (columns unchanged: True)


## 5. `add_feature()`: compute columns

`add_feature()` adds a column derived from the ones you already have and returns a
new RadDB, so it drops straight into a pipeline. The function receives the polars
frame; return a Series, a numpy array, or a polars expression.

In [ ]:
derived = rdf.add_feature("DBZH_lin", lambda df: 10 ** (df["DBZH"] / 10)).add_feature(
    "DBZH_dev",
    lambda df: df["DBZH"] - df["DBZH"].mean(),
)
derived.head()

gate_id,time,DBZH,ZDR,KDP,RHOHV,PHIDP,TEMP,volume_time,radar,DBZH_lin,DBZH_dev
i64,datetime[ns],f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str,f32,f32
713647000000000250,2024-06-01 12:00:08.912507648,1.17,-3.06,0.0,0.405933,98.02301,NaN,2024-06-01 12:00:02 UTC,"""FANJ""",1.309182,-13.046085
713647000000015250,2024-06-01 12:00:08.912507648,6.1,3.24,0.0,0.268918,137.750793,NaN,2024-06-01 12:00:02 UTC,"""FANJ""",4.073803,-8.116085
713647000000015750,2024-06-01 12:00:08.912507648,3.62,1.11,0.0,0.689683,91.090424,NaN,2024-06-01 12:00:02 UTC,"""FANJ""",2.301442,-10.596086
713647000000016250,2024-06-01 12:00:08.912507648,0.1,-2.98,0.0,0.816184,91.442001,NaN,2024-06-01 12:00:02 UTC,"""FANJ""",1.023293,-14.116085
713647000000016750,2024-06-01 12:00:08.912507648,3.73,10.08,0.0,0.509667,102.296822,NaN,2024-06-01 12:00:02 UTC,"""FANJ""",2.360478,-10.486086


If you would rather work in plain polars or pandas, nothing stops you — `.data`
is an ordinary polars frame, and `to_pandas()` gives an ordinary pandas one.

In [ ]:
rdf.data.with_columns((pl.col("DBZH") - pl.col("ZDR")).alias("DIFF"))

df = rdf.to_pandas()
df["DIFF"] = df["DBZH"] - df["ZDR"]

print(f"rdf type: {type(rdf.data)}")
print(f"df  type: {type(df)}")
df.head()

rdf type: <class 'polars.dataframe.frame.DataFrame'>
df  type: <class 'pandas.DataFrame'>


,gate_id,time,DBZH,ZDR,KDP,RHOHV,PHIDP,TEMP,volume_time,radar,DIFF
0,713647000000000250,2024-06-01 12:00:08.912507648,1.17,-3.06,0.0,0.405933,98.023010,NaN,2024-06-01 12:00:02+00:00,FANJ,4.23
1,713647000000015250,2024-06-01 12:00:08.912507648,6.10,3.24,0.0,0.268918,137.750793,NaN,2024-06-01 12:00:02+00:00,FANJ,2.86
2,713647000000015750,2024-06-01 12:00:08.912507648,3.62,1.11,0.0,0.689683,91.090424,NaN,2024-06-01 12:00:02+00:00,FANJ,2.51
3,713647000000016250,2024-06-01 12:00:08.912507648,0.10,-2.98,0.0,0.816184,91.442001,NaN,2024-06-01 12:00:02+00:00,FANJ,3.08
4,713647000000016750,2024-06-01 12:00:08.912507648,3.73,10.08,0.0,0.509667,102.296822,NaN,2024-06-01 12:00:02+00:00,FANJ,-6.35


## 6. Framework converter

The gates can leave RadDB as a pandas DataFrame, a geopandas GeoDataFrame, or an
xarray DataTree — three converters for three different frameworks.

### `to_pandas()`: the DataFrame

`to_pandas()` returns the loaded gates as an ordinary pandas DataFrame. On its own
it hands back exactly what is stored per gate: `gate_id`, `time`, the polarimetric variables, and
the `volume_time` / `radar` labels.

Geometry is **not** stored per gate — it lives once in the LUT — so it is merged
on `gate_id` only when you ask for it:

| call | columns added |
|---|---|
| `to_pandas()` | nothing; the stored columns only (dynamic variables) |
| `to_pandas(with_geometry=True)` | `latitude`, `longitude`, `altitude`, `sweep` |
| `to_pandas(with_polar_coords=True)` | the same, **plus** `range`, `azimuth`, `elevation_angle` |

`with_polar_coords` implies `with_geometry`. The polar coordinates are off by
default because they repeat what the Cartesian columns already say, unless you are
working in polar space.

Note what is **not** included: `x`, `y`, `z` — metres from the radar — are never
added by either flag, and the projected `x_<epsg>` / `y_<epsg>` appear only under a
condition. The next three cells explain why, and how to load all of them.

In [ ]:
# No flags: the stored columns only, exactly as open() loaded them.
df = rain.to_pandas()
print("to_pandas():", type(df).__name__, df.shape)
print(list(df.columns))

to_pandas(): DataFrame (704733, 10)
['gate_id', 'time', 'DBZH', 'ZDR', 'KDP', 'RHOHV', 'PHIDP', 'TEMP', 'volume_time', 'radar']


In [ ]:
# with_geometry=True joins the per-gate coordinates from the LUT on gate_id.
df_geo = rain.to_pandas(with_geometry=True)
print("added by with_geometry     :", [c for c in df_geo.columns if c not in df.columns])

added by with_geometry     : ['latitude', 'longitude', 'altitude', 'sweep']


In [ ]:
# with_polar_coords=True also brings the polar coordinates the geometry came from.
df_polar = rain.to_pandas(with_polar_coords=True)
print("added by with_polar_coords :", [c for c in df_polar.columns if c not in df.columns])
df_polar.head(3)

added by with_polar_coords : ['latitude', 'longitude', 'altitude', 'sweep', 'range', 'azimuth', 'elevation_angle']


,gate_id,time,DBZH,ZDR,KDP,RHOHV,PHIDP,TEMP,volume_time,radar,latitude,longitude,altitude,sweep,range,azimuth,elevation_angle
0,713647000000098250,2024-06-01 12:00:08.912507648,25.190001,-0.09,0.00,0.998367,75.445419,NaN,2024-06-01 12:00:02+00:00,FANJ,61.787348,27.10806,1221.548181,0,98250.0,0.0,0.3
1,713647000000098750,2024-06-01 12:00:08.912507648,24.959999,-0.02,0.01,0.997940,74.786217,NaN,2024-06-01 12:00:02+00:00,FANJ,61.791844,27.10806,1229.962841,0,98750.0,0.0,0.3
2,713647000000099250,2024-06-01 12:00:08.912507648,22.090000,0.02,0.01,0.997162,74.736778,NaN,2024-06-01 12:00:02+00:00,FANJ,61.796339,27.10806,1238.406920,0,99250.0,0.0,0.3


### Where the geometry lives

Two things are easy to trip over:

- **`x_<epsg>` / `y_<epsg>` appear only if the RadDB was created with `crs=`.**
  `crs()` reports the archive's projection either way, but the converters add the
  projected pair only when a projection was asked for explicitly.
- **`x` / `y` / `z`** — metres from the radar — are LUT columns that no converter
  attaches. Join the LUT yourself to get them, or any other LUT column.

In [ ]:
# Projected coordinates: state the CRS when creating the RadDB, and
# with_geometry=True then adds x_<epsg> / y_<epsg> alongside lon/lat/alt.
db_proj = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=3067)
rain_proj = db_proj.open(radars="FANJ", filters={"var": "DBZH", "logic": ">", "threshold": 20})

print("without crs= :", list(rain.to_pandas(with_geometry=True).columns))
print("with crs=3067:", list(rain_proj.to_pandas(with_geometry=True).columns))

without crs= : ['gate_id', 'time', 'DBZH', 'ZDR', 'KDP', 'RHOHV', 'PHIDP', 'TEMP', 'volume_time', 'radar', 'latitude', 'longitude', 'altitude', 'sweep']


with crs=3067: ['gate_id', 'time', 'DBZH', 'ZDR', 'KDP', 'RHOHV', 'PHIDP', 'TEMP', 'volume_time', 'radar', 'latitude', 'longitude', 'altitude', 'sweep', 'x_3067', 'y_3067']


In [ ]:
# Any LUT column can be attached by joining on gate_id.  This is also how you add
# geometry to a frame loaded with open(), and the only way to get x / y / z
# (metres from the radar), which no converter attaches.
geometry = db.get_lut("FANJ").select(["gate_id", "x", "y", "z", "x_3067", "y_3067"])
joined = rain.data.join(geometry, on="gate_id", how="left")
joined.select(["gate_id", "DBZH", "x", "y", "z", "x_3067", "y_3067"]).head()

gate_id,DBZH,x,y,z,x_3067,y_3067
i64,f32,f64,f64,f64,f64,f64
713647000000098250,25.190001,0.0,98238.323898,1082.548181,505699.30037,6.8505e6
713647000000098750,24.959999,0.0,98738.189137,1090.962841,505698.46809,6.8510e6
713647000000099250,22.09,0.0,99238.053384,1099.40692,505697.635776,6.8515e6
713647000000099750,20.93,0.0,99737.916635,1107.880417,505696.803428,6.8520e6
713647000000100250,24.91,0.0,100237.778888,1116.383332,505695.971046,6.8525e6


### `to_geopandas()` — points with a CRS

In [ ]:
# geopandas: point geometry per gate, ready for spatial joins or QGIS
gdf = rain.to_geopandas()
print("to_geopandas: ", type(gdf))
print("CRS:", gdf.crs)
gdf[["gate_id", "DBZH", "geometry"]].head()

to_geopandas:  <class 'geopandas.geodataframe.GeoDataFrame'>
CRS: EPSG:4326


,gate_id,DBZH,geometry
0,713647000000098250,25.190001,POINT (27.10806 61.78735)
1,713647000000098750,24.959999,POINT (27.10806 61.79184)
2,713647000000099250,22.090000,POINT (27.10806 61.79634)
3,713647000000099750,20.930000,POINT (27.10806 61.80083)
4,713647000000100250,24.910000,POINT (27.10806 61.80533)


### `to_datatree()` — back to xarray

In [ ]:
# DataTree: the full polar structure, for xarray workflows.
# A DataTree describes ONE volume: each sweep is an (azimuth x range) grid and
# time is a per-ray coordinate, so there is no dimension to stack volumes along.
# Choose which volume to rebuild; to_datatree() then NaN-fills the gates that
# were filtered out, restoring the complete azimuth x range grid.
volumes = rdf.data["volume_time"].unique().sort().to_list()
print(f"{len(volumes)} volumes loaded -> rebuilding the first one\n")

dt = rdf.to_datatree(timestep=volumes[0])
dt

13 volumes loaded -> rebuilding the first one



<xarray.DataTree>
Group: /
├── Group: /sweep_0
│       Dimensions:          (azimuth: 360, range: 500)
│       Coordinates: (12/15)
│         * azimuth          (azimuth) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
│         * range            (range) float32 2kB 250.0 750.0 ... 2.492e+05 2.498e+05
│           latitude         (azimuth, range) float64 1MB 60.91 60.91 ... 63.14 63.15
│           longitude        (azimuth, range) float64 1MB 27.11 27.11 ... 27.03 27.03
│           altitude         (azimuth, range) float64 1MB 140.3 143.0 ... 5.117e+03
│           x                (azimuth, range) float64 1MB 0.0 0.0 ... -4.357e+03
│           ...               ...
│           y_3067           (azimuth, range) float64 1MB 6.752e+06 ... 7.002e+06
│           site_latitude    float64 8B 60.9
│           site_longitude   float64 8B 27.11
│           site_altitude    float64 8B 139.0
│           sweep_number     int64 8B 0
│           elevation_angle  float64 8B 0.3
│       Data variables:
│           time             (azimuth, range) datetime64[ns] 1MB 2024-06-01T12:00:08....
│           DBZH             (azimuth, range) float32 720kB 1.17 nan nan ... nan nan nan
│           ZDR              (azimuth, range) float32 720kB -3.06 nan nan ... nan nan
│           KDP              (azimuth, range) float32 720kB 0.0 nan nan ... nan nan nan
│           RHOHV            (azimuth, range) float32 720kB 0.4059 nan nan ... nan nan
│           PHIDP            (azimuth, range) float32 720kB 98.02 nan nan ... nan nan
│           TEMP             (azimuth, range) float32 720kB nan nan nan ... nan nan nan
├── Group: /sweep_1
│       Dimensions:          (azimuth: 360, range: 500)
│       Coordinates: (12/15)
│         * azimuth          (azimuth) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
│         * range            (range) float32 2kB 250.0 750.0 ... 2.492e+05 2.498e+05
│           latitude         (azimuth, range) float64 1MB 60.91 60.91 ... 63.14 63.15
│           longitude        (azimuth, range) float64 1MB 27.11 27.11 ... 27.03 27.03
│           altitude         (azimuth, range) float64 1MB 142.1 148.2 ... 6.859e+03
│           x                (azimuth, range) float64 1MB 0.0 0.0 ... -4.356e+03
│           ...               ...
│           y_3067           (azimuth, range) float64 1MB 6.752e+06 ... 7.002e+06
│           site_latitude    float64 8B 60.9
│           site_longitude   float64 8B 27.11
│           site_altitude    float64 8B 139.0
│           sweep_number     int64 8B 1
│           elevation_angle  float64 8B 0.7
│       Data variables:
│           time             (azimuth, range) datetime64[ns] 1MB 2024-06-01T12:00:29....
│           DBZH             (azimuth, range) float32 720kB 0.23 nan nan ... nan nan nan
│           ZDR              (azimuth, range) float32 720kB -6.25 nan nan ... nan nan
│           KDP              (azimuth, range) float32 720kB 0.0 nan nan ... nan nan nan
│           RHOHV            (azimuth, range) float32 720kB 0.4484 nan nan ... nan nan
│           PHIDP            (azimuth, range) float32 720kB 74.04 nan nan ... nan nan
│           TEMP             (azimuth, range) float32 720kB nan nan nan ... nan nan nan
├── Group: /sweep_2
│       Dimensions:          (azimuth: 360, range: 500)
│       Coordinates: (12/15)
│         * azimuth          (azimuth) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
│         * range            (range) float32 2kB 250.0 750.0 ... 2.492e+05 2.498e+05
│           latitude         (azimuth, range) float64 1MB 60.91 60.91 ... 63.14 63.15
│           longitude        (azimuth, range) float64 1MB 27.11 27.11 ... 27.03 27.03
│           altitude         (azimuth, range) float64 1MB 145.5 158.7 ... 1.034e+04
│           x                (azimuth, range) float64 1MB 0.0 0.0 ... -4.353e+03
│           ...               ...
│           y_3067           (azimuth, range) float64 1MB 6.752e+06 ... 7.002e+06
│           site_latitude    float64 8B 60.9
│           site

---
**Next:** [3 — Areas of interest](03_area_of_interest.ipynb)